# AI Portfolio Analyzer — Colab Training Notebook

Train the LSTM / Transformer model on Google Colab (free GPU).

**Steps:**
1. Verify GPU
2. Mount Google Drive
3. Upload/unzip project files
4. Copy to /content for fast I/O
5. Install dependencies
6. (Optional) W&B login
7. Load config & configure for Colab
8. Download raw market data
9. Build processed feature datasets
10. Train model
11. Evaluate on test set
12. Visualise predictions
13. Run live inference
14. Run backtest
15. Plot equity curve
16. Save checkpoint to Drive

> Educational use only — not financial advice.

## Cell 1 — Check GPU

Go to **Runtime → Change runtime type → T4 GPU** before running if you see 'No GPU detected'.

In [ ]:
# Cell 1: Verify GPU
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU detected. Go to Runtime -> Change runtime type -> GPU (T4 recommended)')

try:
    import torch
    print(f'PyTorch version : {torch.__version__}')
    print(f'CUDA available  : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU             : {torch.cuda.get_device_name(0)}')
        print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('PyTorch not yet installed — will be installed in Cell 5.')


## Cell 2 — Mount Google Drive

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ai-portfolio-analyzer'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')


## Cell 3 — Upload project files

**Option A — Upload the zip (recommended for first run):**
Upload `ai-portfolio-analyzer.zip` to your Google Drive root, then run this cell.

**Option B — Already unzipped in Drive:** skip this cell.

**Option C — Clone from GitHub:**
`!git clone <your-repo-url> /content/drive/MyDrive/ai-portfolio-analyzer`

In [ ]:
# Cell 3: Unzip project from Drive (skip if already unzipped)
import os

ZIP_PATH = '/content/drive/MyDrive/ai-portfolio-analyzer.zip'

if os.path.exists(ZIP_PATH):
    os.system(f'unzip -q -o "{ZIP_PATH}" -d /content/drive/MyDrive/')
    print('Unzipped project to Drive.')
elif os.path.isdir(PROJECT_DIR) and os.listdir(PROJECT_DIR):
    print('Project folder already exists in Drive — skipping unzip.')
else:
    print('Zip not found at:', ZIP_PATH)
    print('Upload ai-portfolio-analyzer.zip to your Drive root, then re-run.')


## Cell 4 — Copy project to /content (fast I/O)

In [ ]:
# Cell 4: Copy project to /content for fast I/O
import shutil, sys, os

CONTENT_DIR = '/content/ai-portfolio-analyzer'

if not os.path.exists(CONTENT_DIR):
    shutil.copytree(PROJECT_DIR, CONTENT_DIR)
    print(f'Copied project -> {CONTENT_DIR}')
else:
    print(f'Project already at {CONTENT_DIR} (skipping copy)')

os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)
print('Working dir:', os.getcwd())


## Cell 5 — Install dependencies

Takes ~3-5 min on first run.

In [ ]:
# Cell 5: Install all dependencies
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'scikit-learn>=1.5', 'scipy>=1.14',
    'yfinance>=0.2.40',
    'torch>=2.1.0',
    'xgboost>=2.0', 'lightgbm>=4.0',
    'transformers>=4.38.0', 'accelerate>=0.27.0', 'sentencepiece>=0.1.99',
    'cvxpy>=1.4',
    'plotly>=5.18.0', 'streamlit>=1.32.0',
    'wandb>=0.16.0',
    'pyyaml>=6.0', 'python-dotenv>=1.0',
    'requests>=2.31.0', 'feedparser>=6.0.10',
    'pyarrow>=14.0', 'fastparquet',
], check=True)
print('All packages installed.')


## Cell 6 — (Optional) W&B login

Get your API key at [wandb.ai/authorize](https://wandb.ai/authorize).

In [ ]:
# Cell 6: W&B login (optional)
USE_WANDB = False  # set True if you have a W&B account

if USE_WANDB:
    import wandb
    wandb.login()  # prompts for API key
else:
    import os
    os.environ['WANDB_DISABLED'] = 'true'
    print('Skipping W&B. Set USE_WANDB=True to enable experiment tracking.')


## Cell 7 — Load config & configure for Colab

Overrides are applied on top of `configs/config.yaml`.
Sentiment fetching is disabled during training (no API keys needed).
Re-enable it later once you have NewsAPI / Finnhub keys in your `.env`.

In [ ]:
# Cell 7: Load config and patch for Colab
import os
from src.utils import load_config, set_seed, detect_device

cfg = load_config('configs/config.yaml')

# --- Model overrides (reduce for free-tier Colab; increase for paid GPU) ---
cfg['model']['type']            = 'lstm'   # 'lstm' | 'transformer' | 'xgboost' | 'lightgbm'
cfg['model']['hidden_size']     = 128      # default 256 — lower if you hit OOM
cfg['model']['num_layers']      = 2        # default 4
cfg['model']['dropout']         = 0.3
cfg['model']['use_uncertainty'] = True

# --- Training overrides ---
cfg['training']['epochs']                 = 60
cfg['training']['batch_size']             = 128
cfg['training']['learning_rate']          = 0.001
cfg['training']['early_stopping_patience'] = 15
cfg['training']['checkpoint_dir']         = 'checkpoints'

# --- Data paths ---
cfg['data']['raw_dir']       = 'data/raw'
cfg['data']['processed_dir'] = 'data/processed'

# --- Disable live sentiment (no API key needed for training) ---
cfg['features']['use_sentiment'] = False

# --- W&B ---
cfg['wandb']['enabled'] = USE_WANDB

# --- Ensure output dirs exist ---
for d in ['data/raw', 'data/processed', 'checkpoints', 'reports/figures']:
    os.makedirs(d, exist_ok=True)

device = detect_device()
set_seed(cfg['training']['seed'])

print(f"Device : {device}")
print(f"Model  : {cfg['model']['type']}  "
      f"hidden={cfg['model']['hidden_size']}  layers={cfg['model']['num_layers']}")
print(f"Epochs : {cfg['training']['epochs']}  "
      f"Batch: {cfg['training']['batch_size']}  "
      f"LR: {cfg['training']['learning_rate']}")


## Cell 8 — Download raw market data

Downloads OHLCV data from Yahoo Finance via `yfinance` for all tickers in the config.
Saves cached `.parquet` files to `data/raw/`.
If the files already exist (e.g. from the uploaded zip), they load from disk — no re-download needed.

In [ ]:
# Cell 8: Download raw market data
from src.dataset import download_all

raw_data = download_all(cfg)

print(f'\n{len(raw_data)} tickers loaded:')
for ticker, df in sorted(raw_data.items()):
    print(f'  {ticker:6s}  {len(df):5,d} rows  '
          f'{df.index[0].date()} -> {df.index[-1].date()}')


## Cell 9 — Build processed feature datasets

Computes technical indicators (RSI, MACD, Bollinger Bands, ATR, rolling means/stds) and saves one `.parquet` per ticker to `data/processed/`.

Set `TRAIN_TICKER = 'ALL'` to process every ticker (slower but trains a richer model).

In [ ]:
# Cell 9: Build processed feature datasets
from src.dataset import build_processed_dataset

TRAIN_TICKER = 'AAPL'  # change to any ticker in cfg['data']['tickers'], or 'ALL'

if TRAIN_TICKER == 'ALL':
    for t in cfg['data']['tickers']:
        try:
            df_p = build_processed_dataset(t, cfg, sentiment_df=None)
            print(f'  {t:6s}  {df_p.shape}')
        except Exception as e:
            print(f'  {t:6s}  ERROR: {e}')
    print('\nAll tickers processed.')
else:
    df_proc = build_processed_dataset(TRAIN_TICKER, cfg, sentiment_df=None)
    print(f'Processed dataset shape: {df_proc.shape}')
    first_cols = list(df_proc.columns[:8])
    extra = len(df_proc.columns) - 8
    print(f'Columns: {first_cols} ... (+{extra} more)')
    print(df_proc.describe().T[['mean', 'std', 'min', 'max']].round(4))


## Cell 10 — Train the model

Runs the full training pipeline:
- Time-series safe train/val/test split (no lookahead)
- StandardScaler fitted on train data only
- LSTM/Transformer with early stopping
- Best checkpoint saved to `checkpoints/best_model.pt`

In [ ]:
# Cell 10: Train model
from src.train import train

ticker_arg = None if TRAIN_TICKER == 'ALL' else TRAIN_TICKER

label = 'all tickers' if ticker_arg is None else ticker_arg
print(f"Training {cfg['model']['type'].upper()} on {label} ...")

train(cfg, ticker=ticker_arg)
print('\nTraining complete! Checkpoint saved to checkpoints/best_model.pt')


## Cell 11 — Evaluate on test set

In [ ]:
# Cell 11: Evaluate model on held-out test set
from src.evaluate import evaluate

eval_ticker = TRAIN_TICKER if TRAIN_TICKER != 'ALL' else cfg['data']['tickers'][0]

metrics, y_pred, y_true = evaluate(cfg, ticker=eval_ticker)

print(f'\nTest Set Metrics - {eval_ticker}')
print('-' * 45)
for k, v in sorted(metrics.items()):
    print(f'  {k:32s}: {v:.4f}')


## Cell 12 — Visualise predictions vs actuals

In [ ]:
# Cell 12: Plot predicted vs actual returns
import plotly.graph_objects as go

n = min(100, len(y_true))
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_true[:n, 0], name='True Return',
                         line=dict(color='#8b949e')))
fig.add_trace(go.Scatter(y=y_pred[:n, 0], name='Pred Return',
                         line=dict(color='#00d4aa')))
fig.update_layout(
    title=f'{eval_ticker} - Return Prediction (first {n} test samples)',
    template='plotly_dark', height=380,
    xaxis_title='Sample', yaxis_title='Log Return',
)
fig.show()


## Cell 13 — Live inference

Fetches the latest market data and runs the trained model to produce a risk/return outlook.

In [ ]:
# Cell 13: Run live inference on latest data
from src.inference import run_inference

result = run_inference(eval_ticker, cfg)

print(f"Ticker           : {result['ticker']}")
print(f"Current Price    : ${result.get('current_price', 0):.2f}")
print(f"Predicted Return : {result.get('pred_return', 0)*100:+.2f}%")
print(f"Predicted Vol    : {result.get('pred_volatility', 0)*100:.2f}%")
print(f"Downside Prob    : {result.get('pred_downside_prob', 0)*100:.1f}%")
print(f"Risk Score       : {result.get('risk_score', 0):.0f} / 100  [{result.get('risk_label')}]")
print(f"Outlook          : {result.get('outlook')}")
sf = result.get('sentiment_features', {})
print(f"Sentiment        : {sf.get('weighted_sentiment', 0):+.3f}  ({sf.get('news_volume', 0)} articles)")


## Cell 14 — Backtest

In [ ]:
# Cell 14: Run portfolio backtest
from src.backtest import run_backtest

bt_result = run_backtest(cfg)
m = bt_result['metrics']

print('Backtest Results:')
print(f"  CAGR          : {m.get('cagr', 0)*100:.2f}%")
print(f"  Sharpe Ratio  : {m.get('sharpe_ratio', 0):.3f}")
print(f"  Sortino Ratio : {m.get('sortino_ratio', 0):.3f}")
print(f"  Max Drawdown  : {m.get('max_drawdown', 0)*100:.2f}%")
print(f"  Win Rate      : {m.get('win_rate', 0)*100:.1f}%")
if 'benchmark_cagr' in m:
    print(f"  Benchmark CAGR: {m['benchmark_cagr']*100:.2f}%")


## Cell 15 — Plot equity curve

In [ ]:
# Cell 15: Plot equity curve vs benchmark
from dashboard.dashboard_utils import backtest_chart

ec = bt_result.get('equity_curve')
bm = bt_result.get('benchmark_curve')

if ec is not None:
    fig = backtest_chart(ec, bm)
    if fig:
        fig.show()
else:
    print('No equity curve available.')


## Cell 16 — Save checkpoints to Drive

Copies `checkpoints/best_model.pt` and `checkpoints/scaler.pkl` back to Google Drive so they survive the Colab session ending.

In [ ]:
# Cell 16: Save checkpoints to Google Drive
import shutil, os

DRIVE_CKPT = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

saved = []
for fname in os.listdir('checkpoints'):
    src_path = os.path.join('checkpoints', fname)
    dst_path = os.path.join(DRIVE_CKPT, fname)
    shutil.copy2(src_path, dst_path)
    saved.append(fname)

print(f'Saved {len(saved)} checkpoint file(s) to Drive:')
for f in saved:
    print(f'  -> {f}')
